# Double Zernike Analysis (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-04-12  
**Last Modified:** 2026-04-12  
**Status:** Draft  
**Keywords:** Double Zernike, AOS, intrinsics, correlations, LUT, rotator, elevation, thermal  

## Description

Aggregate Double Zernike (DZ) fit results across all pipeline chunks (OCS or CCS
coordinate system) and run cross-coefficient and operational analyses.

Key functionality:
1. Load and concatenate all `*_fits.parquet` (OCS) or `*_ccs_fits.parquet` (CCS)
   from the pipeline `output/` tree.
2. Reproduce DZ-vs-thermal scatter plots from `intrinsics_thermal_correlations`.
3. Full pairwise correlation analysis over all DZ coefficients; pull out the
   10 largest off-diagonal correlations and make scatter plots.
4. LUT exploration: DZ terms vs rotator angle (elevation ∈ 70 ± 3 deg) and
   DZ terms vs elevation (rotator angle ∈ 0 ± 3 deg).

**Output:** PDF in `output/dz_analysis_{coord_sys}.pdf`  
**Based on:** intrinsics_thermal_correlations.ipynb, run_pipeline outputs

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-12 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Thermal Correlations](#thermal)
6. [Full DZ Correlation Analysis](#corr)
7. [LUT: DZ vs Rotator and Elevation](#lut)
8. [Output](#output)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Coordinate system for fit tables: 'OCS' or 'CCS'
coord_sys = 'OCS'

# Where to look for fit parquet files (relative to this notebook)
output_root = 'output'

# Glob patterns: OCS uses *_fits.parquet, CCS uses *_ccs_fits.parquet
fit_glob = {
    'OCS': '*[!_ccs]_fits.parquet',
    'CCS': '*_ccs_fits.parquet',
}[coord_sys]

# DZ fit prefix to use for analysis (z1toz3 or z1toz6)
dz_prefix = 'z1toz6'

# Quality cuts
max_coeff_um = 2.0  # reject visits with any DZ coefficient > this (µm)

# Thermal variables (only used if present in the merged tables)
thermal_vars = [
    'cam_air_temp', 'm2_air_temp', 'm1m3_air_temp', 'outside_temp',
    'm2_delta_t', 'cam_m1m3_delta_t', 'dome_delta_t',
    'x_gradient', 'y_gradient', 'z_gradient', 'radial_gradient',
    'tma_truss_temp_pxpy', 'tma_truss_temp_mxmy',
]

# DZ terms to plot vs thermal (k = focal Noll, j = pupil Noll)
dz_terms_for_thermal = [
    (5, 5), (6, 6), (4, 4), (5, 10), (6, 9),
]

# Top-N largest correlations to scatter-plot
top_n_corr = 10

# LUT slice tolerances
elev_center_for_rotator_scan = 70.0   # degrees
elev_tol = 3.0
rot_center_for_elev_scan = 0.0        # degrees
rot_tol = 3.0

# Output PDF
output_pdf = f'{output_root}/dz_analysis_{coord_sys}.pdf'

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

<a id='functions'></a>
## Helper Functions

In [ ]:
def find_fit_files(output_root, coord_sys):
    """Locate all DZ fit parquet files for the requested coord system.

    OCS:  files ending in '_fits.parquet' but NOT '_ccs_fits.parquet'.
    CCS:  files ending in '_ccs_fits.parquet'.
    """
    root = Path(output_root)
    all_fits = sorted(root.glob('*_fits.parquet'))
    if coord_sys == 'CCS':
        return [p for p in all_fits if p.name.endswith('_ccs_fits.parquet')]
    return [p for p in all_fits if not p.name.endswith('_ccs_fits.parquet')]


def load_all_fits(files, source_label='source'):
    """Load and concatenate fit parquet tables, tagging each row with its source."""
    frames = []
    for p in files:
        df = pd.read_parquet(p)
        df[source_label] = p.stem
        frames.append(df)
        print(f"  {p.name}: {len(df)} rows, {len(df.columns)} cols")
    if not frames:
        raise FileNotFoundError('No fit parquet files found')
    combined = pd.concat(frames, ignore_index=True, sort=False)
    return combined


def dz_coeff_columns(df, prefix):
    """Return all DZ coefficient columns for a given prefix.

    Columns are named '{prefix}_z{j}_c{k}' where j = pupil Noll, k = focal Noll.
    Excludes '_err', '_scale', '_bad_fit'.
    """
    pattern = re.compile(rf'^{re.escape(prefix)}_z\d+_c\d+$')
    return [c for c in df.columns if pattern.match(c)]


def parse_jk(col, prefix):
    """Parse '(j, k)' from a column like 'z1toz6_z9_c5'."""
    m = re.match(rf'^{re.escape(prefix)}_z(\d+)_c(\d+)$', col)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))


def short_label(col, prefix):
    j, k = parse_jk(col, prefix)
    return f'(k={k}, j={j})'


def quality_cut(df, prefix, max_coeff_um):
    """Apply standard quality cuts: drop bad_fit and outliers."""
    n0 = len(df)
    bad_col = f'{prefix}_bad_fit'
    if bad_col in df.columns:
        df = df[~df[bad_col].astype(bool)].copy()
    elif 'bad_fit' in df.columns:
        df = df[~df['bad_fit'].astype(bool)].copy()
    cols = dz_coeff_columns(df, prefix)
    out_mask = df[cols].abs().gt(max_coeff_um).any(axis=1)
    df = df[~out_mask].copy()
    print(f'Quality cut: {len(df)}/{n0} rows kept '
          f'(prefix={prefix}, |c| < {max_coeff_um} µm)')
    return df

<a id='data'></a>
## Data Access

In [ ]:
fit_files = find_fit_files(output_root, coord_sys)
print(f'Found {len(fit_files)} {coord_sys} fit files in {output_root}/')
for p in fit_files:
    print(f'  {p}')

df_all = load_all_fits(fit_files, source_label='chunk')
print(f'\nCombined: {len(df_all)} rows x {len(df_all.columns)} columns')
print(f'day_obs range: {df_all["day_obs"].min()} - {df_all["day_obs"].max()}')

In [ ]:
df = quality_cut(df_all, dz_prefix, max_coeff_um)
dz_cols = dz_coeff_columns(df, dz_prefix)
print(f'Number of DZ coefficient columns: {len(dz_cols)}')

# Sanity check on key metadata columns
for col in ['rotator_angle', 'alt', 'az'] + thermal_vars:
    n_valid = df[col].notna().sum() if col in df.columns else 0
    flag = 'OK' if col in df.columns else 'MISSING'
    print(f'  {col:<25s} {flag:<8s} valid={n_valid}/{len(df)}')

<a id='thermal'></a>
## Thermal Correlations

Reproduce the DZ-vs-thermal scatter plots from
`intrinsics_thermal_correlations.ipynb` over the combined multi-chunk dataset.
Each page shows one DZ coefficient against every thermal variable, with a
linear fit line and Pearson correlation `r`.

In [ ]:
def thermal_scatter_page(df, dz_col, k, j, thermal_vars):
    """Make a page of scatter plots: dz_col vs every thermal variable."""
    have = [tv for tv in thermal_vars if tv in df.columns]
    n = len(have)
    ncols = 4
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.2 * nrows))
    fig.suptitle(
        f'DZ (k={k}, j={j}) = {dz_col}  vs thermal variables',
        fontsize=13, y=1.01)
    axes = np.atleast_1d(axes).ravel()
    for idx, tv in enumerate(have):
        ax = axes[idx]
        m = df[dz_col].notna() & df[tv].notna()
        x = df.loc[m, tv].values
        y = df.loc[m, dz_col].values
        ax.scatter(x, y, s=10, alpha=0.6, edgecolors='none')
        if len(x) > 2:
            c = np.polyfit(x, y, 1)
            r = np.corrcoef(x, y)[0, 1]
            xf = np.linspace(x.min(), x.max(), 50)
            ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.2, alpha=0.8)
            ax.text(0.05, 0.92, f'r = {r:.3f}', transform=ax.transAxes,
                    fontsize=9, va='top',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8))
        ax.set_xlabel(tv, fontsize=8)
        ax.set_ylabel(f'(k={k},j={j}) [µm]', fontsize=8)
        ax.tick_params(labelsize=7)
    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)
    fig.tight_layout()
    return fig

In [ ]:
thermal_figs = []
for k, j in dz_terms_for_thermal:
    col = f'{dz_prefix}_z{j}_c{k}'
    if col not in df.columns:
        print(f'Skipping {col}: not in table')
        continue
    if not any(tv in df.columns for tv in thermal_vars):
        print('No thermal variables present in tables — skipping section.')
        break
    fig = thermal_scatter_page(df, col, k, j, thermal_vars)
    thermal_figs.append(fig)
    plt.show()

<a id='corr'></a>
## Full DZ Correlation Analysis

Compute the pairwise Pearson correlation matrix over all DZ coefficients,
display it as a heatmap, then extract the `top_n_corr` largest off-diagonal
correlations (by `|r|`) and scatter-plot each pair.

In [ ]:
corr_df = df[dz_cols].dropna()
print(f'Correlation sample size: {len(corr_df)} visits, {len(dz_cols)} DZ terms')
corr_matrix = corr_df.corr(method='pearson')

fig_corr, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
labels = [short_label(c, dz_prefix) for c in dz_cols]
ax.set_xticks(range(len(dz_cols)))
ax.set_xticklabels(labels, rotation=90, fontsize=6)
ax.set_yticks(range(len(dz_cols)))
ax.set_yticklabels(labels, fontsize=6)
ax.set_title(f'Pearson correlation — all {dz_prefix} DZ coefficients ({coord_sys})')
plt.colorbar(im, ax=ax, shrink=0.8, label='r')
fig_corr.tight_layout()
plt.show()

In [ ]:
# Extract upper-triangular off-diagonal correlations
n = len(dz_cols)
iu, ju = np.triu_indices(n, k=1)
rs = corr_matrix.values[iu, ju]
order = np.argsort(np.abs(rs))[::-1]
top = order[:top_n_corr]

top_pairs = []
print(f'Top {top_n_corr} |r| pairs:')
print(f'{"#":>3} {"col_a":<22} {"col_b":<22} {"r":>7}')
for rank, idx in enumerate(top, 1):
    a = dz_cols[iu[idx]]
    b = dz_cols[ju[idx]]
    r = rs[idx]
    top_pairs.append((a, b, r))
    print(f'{rank:>3} {a:<22} {b:<22} {r:>7.3f}')

In [ ]:
# Scatter plots of the top pairs in a grid
ncols = 2
nrows = int(np.ceil(top_n_corr / ncols))
fig_top, axes = plt.subplots(nrows, ncols, figsize=(11, 4.2 * nrows))
axes = np.atleast_1d(axes).ravel()
for idx, (a, b, r) in enumerate(top_pairs):
    ax = axes[idx]
    m = df[a].notna() & df[b].notna()
    x = df.loc[m, a].values
    y = df.loc[m, b].values
    ax.scatter(x, y, s=12, alpha=0.6, edgecolors='none')
    c = np.polyfit(x, y, 1)
    xf = np.linspace(x.min(), x.max(), 50)
    ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.2, alpha=0.8)
    ax.set_xlabel(f'{a}\n{short_label(a, dz_prefix)} [µm]', fontsize=9)
    ax.set_ylabel(f'{b}\n{short_label(b, dz_prefix)} [µm]', fontsize=9)
    ax.set_title(f'r = {r:+.3f}   (n={m.sum()})', fontsize=10)
for idx in range(len(top_pairs), len(axes)):
    axes[idx].set_visible(False)
fig_top.suptitle(
    f'Top {top_n_corr} DZ–DZ correlations ({dz_prefix}, {coord_sys})',
    fontsize=13, y=1.005)
fig_top.tight_layout()
plt.show()

<a id='lut'></a>
## LUT: DZ vs Rotator and Elevation

Inputs to a future Look-Up Table for AOS feed-forward correction.

* **Rotator scan** — plot every DZ coefficient vs `rotator_angle` for visits
  with `alt` within ±3 deg of 70 deg.
* **Elevation scan** — plot every DZ coefficient vs `alt` for visits with
  `rotator_angle` within ±3 deg of 0 deg.

In [ ]:
def grid_scan_page(df, x_col, dz_cols, prefix, title, x_label, x_unit='deg'):
    """Plot every DZ coefficient vs x_col in a tight grid."""
    n = len(dz_cols)
    ncols = 6
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(2.3 * ncols, 1.9 * nrows),
                             sharex=True)
    axes = np.atleast_1d(axes).ravel()
    for idx, col in enumerate(dz_cols):
        ax = axes[idx]
        m = df[col].notna() & df[x_col].notna()
        x = df.loc[m, x_col].values
        y = df.loc[m, col].values
        ax.scatter(x, y, s=6, alpha=0.6, edgecolors='none')
        ax.axhline(0, color='gray', lw=0.5, alpha=0.6)
        ax.set_title(short_label(col, prefix), fontsize=7)
        ax.tick_params(labelsize=6)
    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)
    fig.suptitle(title, fontsize=12, y=1.005)
    fig.supxlabel(f'{x_label} [{x_unit}]', fontsize=10)
    fig.supylabel('DZ coefficient [µm]', fontsize=10)
    fig.tight_layout()
    return fig

In [ ]:
# DZ vs rotator angle, at fixed elevation band
elev_lo = elev_center_for_rotator_scan - elev_tol
elev_hi = elev_center_for_rotator_scan + elev_tol
mask_elev = df['alt'].between(elev_lo, elev_hi)
df_rot = df[mask_elev & df['rotator_angle'].notna()].copy()
print(f'Elevation slice {elev_lo:.0f}–{elev_hi:.0f} deg: '
      f'{len(df_rot)}/{len(df)} visits')

if len(df_rot) >= 5:
    fig_rot = grid_scan_page(
        df_rot, 'rotator_angle', dz_cols, dz_prefix,
        title=(f'DZ vs rotator angle  (elevation = '
               f'{elev_center_for_rotator_scan:.0f} ± {elev_tol:.0f} deg, '
               f'n={len(df_rot)}, {coord_sys})'),
        x_label='rotator angle')
    plt.show()
else:
    fig_rot = None
    print('Too few visits in elevation slice for plot.')

In [ ]:
# DZ vs elevation, at fixed rotator-angle band
rot_lo = rot_center_for_elev_scan - rot_tol
rot_hi = rot_center_for_elev_scan + rot_tol
mask_rot = df['rotator_angle'].between(rot_lo, rot_hi)
df_elev = df[mask_rot & df['alt'].notna()].copy()
print(f'Rotator slice {rot_lo:.0f}–{rot_hi:.0f} deg: '
      f'{len(df_elev)}/{len(df)} visits')

if len(df_elev) >= 5:
    fig_elev = grid_scan_page(
        df_elev, 'alt', dz_cols, dz_prefix,
        title=(f'DZ vs elevation  (rotator = '
               f'{rot_center_for_elev_scan:.0f} ± {rot_tol:.0f} deg, '
               f'n={len(df_elev)}, {coord_sys})'),
        x_label='elevation (alt)')
    plt.show()
else:
    fig_elev = None
    print('Too few visits in rotator slice for plot.')

<a id='output'></a>
## Output

Combine all figures into a single PDF.

In [ ]:
Path(output_pdf).parent.mkdir(parents=True, exist_ok=True)

with PdfPages(output_pdf) as pdf:
    for fig in thermal_figs:
        pdf.savefig(fig, bbox_inches='tight')
    pdf.savefig(fig_corr, bbox_inches='tight')
    pdf.savefig(fig_top, bbox_inches='tight')
    if fig_rot is not None:
        pdf.savefig(fig_rot, bbox_inches='tight')
    if fig_elev is not None:
        pdf.savefig(fig_elev, bbox_inches='tight')

print(f'Wrote PDF: {output_pdf}')